In [68]:
import openai
import instructor
import tiktoken
import pandas as pd
import numpy as np

from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Filter,
    FieldCondition,
    FusionQuery,
    MatchValue,
    MatchAny,
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    Document,
    PointStruct,
    Prefetch,
    FusionQuery,
    RrfQuery,
    Rrf
)

### Qdrant collection for Hybrid Search

In [34]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [35]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    }
)

True

In [36]:
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [42]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [43]:
def get_embedding_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Retreive all item IDs from Amazon Items Qdrant Collection

### Read the sampled dataset with Amazon inventory data

In [19]:
data_reviews = pd.read_json("../../data/Books_2022_2023_with_category_ratings_10_sample_1000.jsonl", lines=True)

In [20]:
data_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,"Fits it's description as a ""biopedia.""",As a lifelong fan of Leon Russell and his musi...,[],1662431155,1662431155,AGKWDJVUMB2EI7U7QHNQWCDH6BDQ,2021-08-05 17:59:11.789,5,True
1,5,Good Devotional for Couples Looking to Connect...,My wife and I have been looking for ways to co...,[],1638787158,1638787158,AFLX66DKF6R3H6OEOC3TIVAYXZIQ,2022-08-24 10:08:38.564,0,False
2,5,A real Sophie’s Choice dilemma,This psychological suspense had a Sophie’s Cho...,[],1728250366,1728250366,AFAYQWKMEVX7GKQ7GZNQXJ7ORLWA,2022-05-04 01:00:59.079,0,False
3,5,A Much Needed Cookbook for Fall and Winter,So many good recipes in this book. The book is...,[],1621458288,1621458288,AE46NJ5H3EJIEHPS3CMZSHQISACA,2022-12-13 00:21:08.059,0,True
4,1,Fake,How is possible that a person can imitate Anth...,[],B0BMJQ2K4B,B0BMJQ2K4B,AH4UCXO5DN4ISACGJEEBGHB35U7Q,2022-12-20 13:25:25.645,9,False


### Functions for pre-processing reviews

In [27]:
def process_reviews(row):
    return f"{row['title']} {row['text']}"

In [28]:
def token_count(row, model="text-embedding-3-small"):
    
    encoding = tiktoken.encoding_for_model(model)
    
    return len(encoding.encode(row["preprocessed_data"]))

In [29]:
data_reviews['preprocessed_data'] = data_reviews.apply(process_reviews, axis=1)
data_reviews['token_count'] = data_reviews.apply(token_count, axis=1)

In [30]:
data_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,token_count
0,5,"Fits it's description as a ""biopedia.""",As a lifelong fan of Leon Russell and his musi...,[],1662431155,1662431155,AGKWDJVUMB2EI7U7QHNQWCDH6BDQ,2021-08-05 17:59:11.789,5,True,"Fits it's description as a ""biopedia."" As a li...",48
1,5,Good Devotional for Couples Looking to Connect...,My wife and I have been looking for ways to co...,[],1638787158,1638787158,AFLX66DKF6R3H6OEOC3TIVAYXZIQ,2022-08-24 10:08:38.564,0,False,Good Devotional for Couples Looking to Connect...,364
2,5,A real Sophie’s Choice dilemma,This psychological suspense had a Sophie’s Cho...,[],1728250366,1728250366,AFAYQWKMEVX7GKQ7GZNQXJ7ORLWA,2022-05-04 01:00:59.079,0,False,A real Sophie’s Choice dilemma This psychologi...,46
3,5,A Much Needed Cookbook for Fall and Winter,So many good recipes in this book. The book is...,[],1621458288,1621458288,AE46NJ5H3EJIEHPS3CMZSHQISACA,2022-12-13 00:21:08.059,0,True,A Much Needed Cookbook for Fall and Winter So ...,32
4,1,Fake,How is possible that a person can imitate Anth...,[],B0BMJQ2K4B,B0BMJQ2K4B,AH4UCXO5DN4ISACGJEEBGHB35U7Q,2022-12-20 13:25:25.645,9,False,Fake How is possible that a person can imitate...,50


In [31]:
len(data_reviews)

6390

In [32]:
data_reviews = data_reviews[data_reviews["token_count"] < 8192]
len(data_reviews)

6390

In [33]:
data_reviews["token_count"].sum()

np.int64(733284)

### Preprocess Reviews data

In [38]:
data_to_embed_review = data_reviews[["preprocessed_data", "parent_asin"]].to_dict(orient="records")

In [39]:
len(data_to_embed_review)

6390

### Embed Data

#### Amazon review data

In [40]:
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_review]

In [44]:
dense_embeddings = get_embedding_batch(text_to_embed_reviews, batch_size=500)

Processed 500 of 6390
Processed 1000 of 6390
Processed 1500 of 6390
Processed 2000 of 6390
Processed 2500 of 6390
Processed 3000 of 6390
Processed 3500 of 6390
Processed 4000 of 6390
Processed 4500 of 6390
Processed 5000 of 6390
Processed 5500 of 6390
Processed 6000 of 6390
Processed 6500 of 6390


In [48]:
pointstructs = []
idx = 1
for embedding, data in zip(dense_embeddings, data_to_embed_review):
    pointstructs.append(
        PointStruct(
            id=idx,
            vector={
                "text-embedding-3-small": embedding
            },
            payload=data
        )
    )
    idx += 1

In [50]:
batch_size_qdrant = 100
counter = 1
total_points_to_upsert = len(pointstructs)
for i in range(0, total_points_to_upsert, batch_size_qdrant):
    batch = pointstructs[i:i+batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True
    )
    print(f"Processed {counter * batch_size_qdrant} of {total_points_to_upsert}")
    counter += 1

Processed 100 of 6390
Processed 200 of 6390
Processed 300 of 6390
Processed 400 of 6390
Processed 500 of 6390
Processed 600 of 6390
Processed 700 of 6390
Processed 800 of 6390
Processed 900 of 6390
Processed 1000 of 6390
Processed 1100 of 6390
Processed 1200 of 6390
Processed 1300 of 6390
Processed 1400 of 6390
Processed 1500 of 6390
Processed 1600 of 6390
Processed 1700 of 6390
Processed 1800 of 6390
Processed 1900 of 6390
Processed 2000 of 6390
Processed 2100 of 6390
Processed 2200 of 6390
Processed 2300 of 6390
Processed 2400 of 6390
Processed 2500 of 6390
Processed 2600 of 6390
Processed 2700 of 6390
Processed 2800 of 6390
Processed 2900 of 6390
Processed 3000 of 6390
Processed 3100 of 6390
Processed 3200 of 6390
Processed 3300 of 6390
Processed 3400 of 6390
Processed 3500 of 6390
Processed 3600 of 6390
Processed 3700 of 6390
Processed 3800 of 6390
Processed 3900 of 6390
Processed 4000 of 6390
Processed 4100 of 6390
Processed 4200 of 6390
Processed 4300 of 6390
Processed 4400 of 63

### Hybrid Retrieval

Retrieve reviews only for the items we are interested in

In [59]:
def retrieve_prefiltered_review_data(query, parent_asins, qdrant_client, k=5):
   
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion='rrf'),
        limit=k
    )

    return results

In [60]:
result = retrieve_prefiltered_review_data(
    "bad quality",
    ["1622775996", "B0C128NPC9", "0593547764", "B0C2S1JJWR"],
    qdrant_client,
    k=20
    )

In [61]:
result.points

[ScoredPoint(id=5715, version=61, score=0.5, payload={'preprocessed_data': "Did not dissapoint I love Greek retellings and this one did not disappoint.  I liked seeing the story of Medusa being told from a different perspective than what we usually get.  I think the author did a great job differentiating each sister's voice, and even if the titles hadn't been the sister's names, you would've been able to distinguish who was narrating.  I did enjoy the overall growth & development of the characters.<br /><br />I did find one of the sisters to be unbearable, and I just disliked her character so much, that it made it hard to read through her chapters, and although I love an advanced vocabulary, I felt distracted by the some of the vocabulary used at times.  Even with not liking the one sister, it was still an entertaining book, and I enjoyed it a whole lot.", 'parent_asin': '0593547764'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4928, version=53, score=0.33333334, p

## Define a reviews tool

In [75]:
def retrieve_prefiltered_review_data(query, parent_asins, qdrant_client, k=5):
   
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion='rrf'),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_data"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores
    }

def process_reviews(reviews):
    formatted_reviews = ""
    for id, review in zip(reviews["retrieved_context_ids"], reviews["retrieved_context"]):
        formatted_reviews += f"- ID: {id}, review: {review}\n"
    
    return formatted_reviews


def get_formatted_item_reviews(query: str, items: List[str], top_k: int = 5) -> str:
    """Get the top k reviews matching a query for a list of prefiltered items.

    Args:
        query: The query to get the top k reviews for
        items: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retreieve, this should be at least 20 if multiple items are prefiltered
    
    Returns:
        A string of the top k reviews with IDs prepending each review. Each line is a single review for one of the items in the items list.
    """

    qdrant_client = QdrantClient(url="http://localhost:6333")
    reviews = retrieve_prefiltered_review_data(query, items, qdrant_client, top_k)

    return process_reviews(reviews)



In [76]:
result = get_formatted_item_reviews("bad quality", ["1622775996", "B0C128NPC9", "0593547764", "B0C2S1JJWR"])

In [77]:
print(result)

- ID: 0593547764, review: Did not dissapoint I love Greek retellings and this one did not disappoint.  I liked seeing the story of Medusa being told from a different perspective than what we usually get.  I think the author did a great job differentiating each sister's voice, and even if the titles hadn't been the sister's names, you would've been able to distinguish who was narrating.  I did enjoy the overall growth & development of the characters.<br /><br />I did find one of the sisters to be unbearable, and I just disliked her character so much, that it made it hard to read through her chapters, and although I love an advanced vocabulary, I felt distracted by the some of the vocabulary used at times.  Even with not liking the one sister, it was still an entertaining book, and I enjoyed it a whole lot.
- ID: 1622775996, review: The life of a true American classical music pioneer, chronicled in detail. I was Margaret Hillis’ young assistant conductor with the Elgin Symphony for three